In [2]:
# %% [markdown]
# # 01_fetch_bird_data.ipynb
# Verifies the BIRD Dev Set (financial domain) is correctly downloaded
# and placed for Phase 2 / Phase 3 experiments.
#
# UNLIKE the Phase 1 fetch notebook, this cannot be fully automated:
# the BIRD dataset is hosted on Alibaba OSS storage
# (bird-bench.oss-cn-beijing.aliyuncs.com), which requires a manual
# browser download. This notebook checks whether the download has
# been placed correctly and gives specific guidance if not, rather
# than attempting to fetch it programmatically.
#
# Official source: https://bird-bench.github.io/
# Direct Dev Set link: https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip
#
# IMPORTANT: Download the "🔥 Dev Set" link specifically -- NOT
# "Train Set" and NOT "Mini-Dev (500)", which are different subsets
# with different question sets. The manuscript's Phase 2 uses the
# standard Dev Set (106 financial-domain questions after filtering).

# %%
import os
import zipfile
import shutil

# %% [markdown]
# ## 1. Check whether the target DB file already exists at the
#    expected path

# %%
EXPECTED_DB_PATH = "./dev/dev_20240627/dev_databases/dev_databases/financial/financial.sqlite"

if os.path.exists(EXPECTED_DB_PATH):
    size_mb = os.path.getsize(EXPECTED_DB_PATH) / (1024 * 1024)
    print(f"✓ Found: {EXPECTED_DB_PATH} ({size_mb:.1f} MB)")
    print("  Nothing further to do -- skip to the verification cells below.")
else:
    print(f"✗ Not found at expected path: {EXPECTED_DB_PATH}")
    print(f"  Resolved absolute path: {os.path.abspath(EXPECTED_DB_PATH)}")
    print()
    print("Searching this directory tree for any financial.sqlite file,")
    print("in case it was extracted to a different location...")
    found_paths = []
    for root, dirs, files in os.walk('.'):
        dirs[:] = [d for d in dirs if d not in ('.git', 'node_modules', '__pycache__')]
        for f in files:
            if f == 'financial.sqlite':
                found_paths.append(os.path.join(root, f))
    if found_paths:
        print(f"\nFound {len(found_paths)} candidate file(s):")
        for p in found_paths:
            print(f"  {p}")
        print("\nIf one of these is the right file, either:")
        print("  (a) move/rename it to match EXPECTED_DB_PATH above, or")
        print("  (b) set DB_PATH in your .env file to point directly to it")
        print("      (utils.py reads DB_PATH from the environment).")
    else:
        print("\nNo financial.sqlite found anywhere in this directory tree.")
        print()
        print("=" * 70)
        print("DOWNLOAD INSTRUCTIONS")
        print("=" * 70)
        print("""
1. In a browser, go to: https://bird-bench.github.io/
   and click the "🔥 Dev Set" link (or use the direct link below).

   Direct link: https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip

   Do NOT use "Train Set" or "Mini-Dev (500)" -- those are different
   question sets.

2. This is a large download (multiple GB, contains all 95 BIRD
   databases across 37 domains, not just financial). Save it as
   dev.zip somewhere on this machine.

3. Extract it. You only need the 'financial' subfolder, but
   extracting the whole archive first is usually simplest.

4. Locate the financial database within the extracted contents --
   look for a folder path containing ".../financial/financial.sqlite"
   somewhere inside the extracted tree. BIRD's zip structure can vary
   slightly between releases, so the exact nesting may differ from
   the historical path this codebase expects.

5. Place (copy or move) that 'financial' folder so the final path is:
     ./dev/dev_20240627/dev_databases/dev_databases/financial/financial.sqlite
   relative to this notebook's working directory.

   If your extracted structure doesn't match this nesting exactly,
   either recreate the folders to match, or update DB_PATH in your
   .env file to point directly at wherever financial.sqlite actually
   ended up -- utils.py will use that instead (see next cell).

6. Re-run this notebook's cells from the top once the file is placed,
   to confirm it's found and structurally valid.
""")

# %% [markdown]
# ## 2. If you placed the file at a DIFFERENT path, check your .env
#    file's DB_PATH setting instead of the hardcoded default above

# %%
from dotenv import load_dotenv
load_dotenv()

env_db_path = os.environ.get("DB_PATH")
if env_db_path:
    print(f".env DB_PATH is set to: {env_db_path}")
    print(f"  Exists: {os.path.exists(env_db_path)}")
    if os.path.exists(env_db_path):
        print("  ✓ This is the path utils.py will actually use.")
else:
    print("No DB_PATH set in .env -- utils.py will fall back to the")
    print(f"default: {EXPECTED_DB_PATH}")

# %% [markdown]
# ## 3. Structural validation: once found, confirm this is really
#    the financial domain DB with the expected tables and the known
#    sensitive columns (A2..A16), not a different/corrupted file

# %%
import sqlite3
import pandas as pd

db_path_to_check = env_db_path if (env_db_path and os.path.exists(env_db_path)) else EXPECTED_DB_PATH

if not os.path.exists(db_path_to_check):
    print(f"Cannot validate -- no file found at '{db_path_to_check}'.")
    print("Complete the download steps above first, then re-run this cell.")
else:
    try:
        conn = sqlite3.connect(db_path_to_check)
        tables = pd.read_sql(
            "SELECT name FROM sqlite_master WHERE type='table'", conn
        )['name'].tolist()
        print(f"Tables found: {sorted(tables)}")

        expected_tables = {'account', 'client', 'disp', 'loan', 'trans', 'card', 'district', 'order'}
        missing_tables = expected_tables - set(tables)
        extra_tables = set(tables) - expected_tables

        if not missing_tables:
            print("✓ All 8 expected tables present.")
        else:
            print(f"✗ Missing expected tables: {missing_tables}")

        # check district table has the A2..A16 sensitive columns
        district_cols = pd.read_sql("PRAGMA table_info('district')", conn)['name'].tolist()
        expected_a_cols = {f'A{i}' for i in range(2, 17)}
        found_a_cols = expected_a_cols & set(district_cols)
        print(f"\nDistrict table sensitive columns found: {len(found_a_cols)} / 15")
        if len(found_a_cols) == 15:
            print("✓ All 15 sensitive columns (A2-A16) present -- this matches")
            print("  the manuscript's RQ2 exposure count exactly.")
        else:
            missing_a = expected_a_cols - found_a_cols
            print(f"✗ Missing: {sorted(missing_a)}")

        # row count sanity checks (BIRD financial DB has known sizes)
        print("\nRow counts (sanity check against known BIRD financial DB sizes):")
        for tbl in ['account', 'client', 'loan', 'trans', 'district']:
            if tbl in tables:
                count = pd.read_sql(f"SELECT COUNT(*) as n FROM {tbl}", conn)['n'][0]
                print(f"  {tbl}: {count:,} rows")

        conn.close()
        print("\n✓ Structural validation complete.")
    except Exception as e:
        print(f"✗ ERROR opening/reading database: {e}")
        print("  The file may be corrupted or not a valid SQLite database")
        print("  (e.g. an incomplete download, or an HTML error page saved")
        print("  with a .sqlite extension). Re-download and try again.")

# %% [markdown]
# ## 4. Confirm the 106-question Phase 2 subset is derivable
#    (manuscript Section 3.3.2: "106 questions against a Czech
#    banking database... simple: 62, moderate: 37, challenging: 7")
#
# BIRD's question/gold-SQL metadata for the dev set is typically
# distributed as a separate JSON file (e.g. dev.json) alongside the
# per-domain databases, containing db_id, question, SQL, evidence,
# and difficulty fields for ALL 95 databases across all domains --
# Phase 2 filters this to db_id == 'financial'.

# %%
# Common locations for the dev.json question file after extraction --
# check a few likely candidates since BIRD's zip layout has varied
# across releases.
candidate_json_paths = [
    "./dev/dev_20240627/dev.json",
    "./dev/dev.json",
    "./dev/dev_20240627/dev_databases/dev.json",
]

found_json = None
for p in candidate_json_paths:
    if os.path.exists(p):
        found_json = p
        break

if found_json:
    import json
    with open(found_json) as f:
        all_questions = json.load(f)
    print(f"Found question metadata at: {found_json}")
    print(f"Total questions across all domains: {len(all_questions)}")

    financial_questions = [q for q in all_questions if q.get('db_id') == 'financial']
    print(f"Financial domain questions: {len(financial_questions)}")
    print(f"  (manuscript states: 106)")

    if len(financial_questions) == 106:
        print("✓ Matches manuscript's stated Phase 2 question count exactly.")
    else:
        print(f"! Count differs from manuscript ({len(financial_questions)} vs 106).")
        print("  BIRD dev set versions have been revised over time (see the")
        print("  'Sept 25, 2023: cleaner version of dev set' and 'Nov 13, 2025:")
        print("  bird-sql-dev-1106 cleaner split' notices on bird-bench.github.io).")
        print("  If this doesn't match, note the discrepancy explicitly in the")
        print("  revision and consider whether the original Phase 2 run used an")
        print("  older dev set version than what's freshly downloaded now.")
else:
    print("Could not locate the question metadata JSON file (dev.json) at any")
    print("of the checked candidate paths:")
    for p in candidate_json_paths:
        print(f"  {p}")
    print()
    print("Search your extracted BIRD download for a file matching 'dev*.json'")
    print("and update candidate_json_paths above with its actual location, or")
    print("place/copy it to one of the checked paths.")

# %% [markdown]
# ## Summary
# Once all cells above show ✓, the BIRD financial database and its
# question metadata are correctly in place for Phase 2 and Phase 3
# experiments (Phase 3 uses the same financial.sqlite with a custom
# question set, not BIRD's own questions, so only the DB file itself
# -- not dev.json -- is required for Phase 3).

✓ Found: ./dev/dev_20240627/dev_databases/dev_databases/financial/financial.sqlite (68.0 MB)
  Nothing further to do -- skip to the verification cells below.
No DB_PATH set in .env -- utils.py will fall back to the
default: ./dev/dev_20240627/dev_databases/dev_databases/financial/financial.sqlite
Tables found: ['account', 'card', 'client', 'disp', 'district', 'loan', 'order', 'trans']
✓ All 8 expected tables present.

District table sensitive columns found: 15 / 15
✓ All 15 sensitive columns (A2-A16) present -- this matches
  the manuscript's RQ2 exposure count exactly.

Row counts (sanity check against known BIRD financial DB sizes):
  account: 4,500 rows
  client: 5,369 rows
  loan: 682 rows
  trans: 1,056,320 rows
  district: 77 rows

✓ Structural validation complete.
Found question metadata at: ./dev/dev_20240627/dev.json
Total questions across all domains: 1534
Financial domain questions: 106
  (manuscript states: 106)
✓ Matches manuscript's stated Phase 2 question count exactly.
